# Automatic Language Detection + AI4Bharat IndicConformer
Uploads audio, detects spoken language with SpeechBrain, then transcribes using `ai4bharat/indic-conformer-600m-multilingual`.

> **Note:** IndicConformer itself does not automatically detect language. This notebook performs language identification first.

In [1]:
!pip -q install -U speechbrain transformers torchaudio librosa soundfile huggingface_hub "onnx==1.20.1" "onnxruntime==1.20.1" "onnxruntime-gpu==1.20.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 32.2 MB/s eta 0:00:00


In [2]:
import torch
import torchaudio
import librosa
import soundfile as sf

from google.colab import userdata, files
from huggingface_hub import login
from transformers import AutoModel
from speechbrain.inference.classifiers import EncoderClassifier


In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


In [4]:
MODEL_NAME="ai4bharat/indic-conformer-600m-multilingual"

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN
).to(device)

print("IndicConformer loaded.")


config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py:   0%|          | 0.00/9.64k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate
IndicConformer loaded.


In [5]:
lid = EncoderClassifier.from_hparams(
    source="speechbrain/lang-id-voxlingua107-ecapa",
    savedir="pretrained_lid",
    run_opts={"device":device},
)
print("Language detector loaded.")


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/lang-id-voxlingua107-ecapa' if not cached


hyperparams.yaml:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/lang-id-voxlingua107-ecapa' if not cached


embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 84.5MB            

embedding_model.ckpt: downloading bytes:           |  0.00B            

...

classifier.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/lang-id-voxlingua107-ecapa' if not cached


label_encoder.txt:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, classifier, label_encoder


Language detector loaded.


In [6]:
LANGUAGE_MAP = {
    "hindi":"hi","english":"en","assamese":"as","bengali":"bn",
    "gujarati":"gu","kannada":"kn","malayalam":"ml","marathi":"mr",
    "nepali":"ne","panjabi":"pa","punjabi":"pa","sanskrit":"sa",
    "sindhi":"sd","tamil":"ta","telugu":"te","urdu":"ur"
}


In [7]:
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

audio, sr = librosa.load(audio_path, sr=16000, mono=True)
sf.write("converted.wav", audio, 16000)

out_prob, score, index, label = lid.classify_file("converted.wav")

detected = label[0].split(': ')[1].lower()
language = LANGUAGE_MAP.get(detected)

print("Detected:", detected)
print("Indic language code:", language)

if language is None:
    raise ValueError(
        f"Detected language '{detected}' is not mapped. "
        "Add it to LANGUAGE_MAP or choose a language manually."
    )

wav, sr = torchaudio.load("converted.wav")
wav = wav.to(device)

print("\nRunning transcription...")

# Depending on the repository version, either __call__ or a custom inference
# method may be used. This follows the published API.
result = model(wav, language, "rnnt")

print("\n===== Transcript =====")
print(result)

Saving Assamese - The Two Roads.mp3 to Assamese - The Two Roads.mp3


Detected: assamese
Indic language code: as

Running transcription...

===== Transcript =====
দুটা ৰাস্তা আছে যীচুৱে শিকাইছিল যে দুটা ৰাস্তা আছে প্ৰত্যেকজন মানুহেই পাপৰ বহল ৰাস্তাৰেহে চলিছে যিয়ে বিনাশলৈ পৰিচালনা কৰে আমি সকলোৱে আদমৰ পৰা উত্তৰাধিকাৰী সূত্ৰে পোৱা পাপতে আছো যীচুৱে কৈছিল যে আমি ঠেক বাটেৰে সোমাব লাগে যি বাটেই ঈশ্বৰৰ স্বৰ্গলৈ পৰিচালনা কৰে যদি এজন ব্যক্তিয়ে এই জীৱনত যীচুক বিশ্বাস কৰে আৰু তেওঁৰ অনুগামী হয় তেওঁ অনন্ত জীৱন পায় তেওঁ বহল ৰাস্তা এৰি ঠেক পটেৰে চলে যি পটে স্বৰ্গলৈ পৰিচালনা কৰে ঠেক পটেৰে চলা সকলৰ কাৰণে আৰু পাপৰ দণ্ডজ্ঞা নাই সুযোগ যদি আপুনি যীচুৰ অনুগামী হব খোজে তেন্তে এনেদৰে ঈশ্বৰৰ সৈতে কথা পাতিব পাৰে সেই ঈশ্বৰ মই স্বীকাৰ কৰিছো যে মই এজন পাপী হওঁ মই বিশ্বাস কৰিছো যে যীচুৱে মোৰ পাপৰ বেজ দিবলৈ খোচত মৃত্যুবৰণ কৰিলে মোক ক্ষমা কৰা আৰু শুদ্ধসূচী কৰা আৰু মোক তোমাৰ পৰিয়ালৰ এজন সদস্য কৰি লোৱা মই যীচুৰ পটৰ অনুগামী হ'ব বিচাৰো আৰু মৃত্যুৰ পিছত মই তোমাৰ সৈতে স্বৰ্গ জীয়াই থাকিব বিচাৰো সেই ঈশ্বৰ ধন্যবাদ


In [8]:
!pip install -q groq

import os
from groq import Groq
from google.colab import userdata

# Retrieve API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

# We'll use the file already converted in the previous steps
filename = "converted.wav"

with open(filename, "rb") as file:
    # Using whisper-large-v3-turbo for faster transcription
    transcription = client.audio.transcriptions.create(
      file=(filename, file.read()),
      model="whisper-large-v3-turbo",
      response_format="verbose_json",
    )

print(f"Detected Language: {transcription.language}")
print("\n===== Groq Transcript =====")
print(transcription.text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.5 MB/s eta 0:00:00
Detected Language: Gujarati

===== Groq Transcript =====
 22 સ્ભી દૂતા રાસ્તા આશે જીસે હિકાયશેલ જે દૂતા રાસ્તા આશે પ્ર્ટેકજન માનુહે પાપર બોહલ રાસ્તારે હે સોલીસે જીએ બીનાખળોલોઈ પરીસલના કરે જીસે કોઈ સ્રજે આમી ઠેક બાતેરે હુમાબલ લાગે જીબાતે ઇસરર સરગોલોઈ પરિશલ ના કરે જોદી એજાણ ભ્યોક્તીએ એજ જિબનાત જીસુક વીસાખ કરે આરુ તેઓર અનુગામી હોય તેઓં અનંત જિબન પાય તેમં બહોલ રાસ્તા એરી ઠેક પતેરે સલે જી પતે સર્ગોલે પરિશલ ના કળે. ફેકપો તેરે સલા હકોલોર કરને આરુ પાપર ડંદગ્યા નાય હુજુક જોદી આપણી જોર અનુગામી હવો ખુજે તેને દેણદે ઇશર હઈટે કતાપાટિબઓ પરે હે ઇશર મે શિકર કરીશુજે મે એજન પાપી હોં મુક હ્યમા કરા આર હુદ્દ્ર હુસીકરા આર મુક તુમાર પર્યાલાર એજં હદેશા કરીલુઓ મોય જીસુર પોટર અનુગામી હવવવવ બીસારુ અર મીટ્તુર પીસત મોય તુમર હયતે સરગો જાય થાગીવ બીસ


In [15]:
!pip install -q sacremoses sentencepiece transformers optimum huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 10.8 MB/s eta 0:00:00


In [32]:
!pip install -q \
transformers \
sentencepiece \
sacremoses \
protobuf \
accelerate

In [ ]:
# Install translation dependencies
!pip install -q sacremoses sentencepiece transformers==4.44.2 optimum

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_IDS = {
    "indic_indic": "ai4bharat/indictrans2-indic-indic-1B",
    "indic_en": "ai4bharat/indictrans2-indic-en-dist-200M",
    "en_indic": "ai4bharat/indictrans2-en-indic-dist-200M",
}

print("Loading IndicTrans2 models...")

tokenizers = {}
translation_models = {}

for name, model_id in MODEL_IDS.items():
    print(f"Loading {model_id}")
    tokenizers[name] = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )
    translation_models[name] = AutoModelForSeq2SeqLM.from_pretrained(
        model_id,
        trust_remote_code=True
    ).to(device)

In [10]:
INDIC_LANGS = {
    "as","bn","brx","doi","gu","hi","kn","ks","kok",
    "mai","ml","mni","mr","ne","or","pa","sa",
    "sat","sd","ta","te","ur"
}

TAG_MAP = {
    "as":"asm_Beng",
    "bn":"ben_Beng",
    "brx":"brx_Deva",
    "doi":"doi_Deva",
    "en":"eng_Latn",
    "gu":"guj_Gujr",
    "hi":"hin_Deva",
    "kn":"kan_Knda",
    "ks":"kas_Arab",
    "kok":"gom_Deva",
    "mai":"mai_Deva",
    "ml":"mal_Mlym",
    "mni":"mni_Beng",
    "mr":"mar_Deva",
    "ne":"npi_Deva",
    "or":"ory_Orya",
    "pa":"pan_Guru",
    "sa":"san_Deva",
    "sat":"sat_Olck",
    "sd":"snd_Arab",
    "ta":"tam_Taml",
    "te":"tel_Telu",
    "ur":"urd_Arab"
}

In [14]:
def translate(text, source_lang, target_lang):
    if source_lang == target_lang:
        return text

    if source_lang == "en" and target_lang in INDIC_LANGS:
        model_key = "en_indic"
    elif source_lang in INDIC_LANGS and target_lang == "en":
        model_key = "indic_en"
    elif source_lang in INDIC_LANGS and target_lang in INDIC_LANGS:
        model_key = "indic_indic"
    else:
        raise ValueError(f"Unsupported translation: {source_lang} -> {target_lang}")

    tokenizer = tokenizers[model_key]
    model = translation_models[model_key]

    src_tag = TAG_MAP[source_lang]
    tgt_tag = TAG_MAP[target_lang]

    inputs = tokenizer(
        text,
        src_lang=src_tag,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            tgt_lang=tgt_tag,
            max_new_tokens=512
        )

    return tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )[0]

print(f"Detected language: {language}")
print(result)

target_language=input("Enter the language to which i need to get translated")

translated_text = translate(result, language, target_language)

print("\n==============================")
print(f"Translation ({language} -> {target_language})")
print("==============================")
print(translated_text)

Detected language: as


KeyboardInterrupt: Interrupted by user

In [11]:
# Install translation dependencies
!pip install -q sacremoses sentencepiece transformers==4.44.2 optimum

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_IDS = {
    "indic_indic": "ai4bharat/indictrans2-indic-indic-1B",
    "indic_en": "ai4bharat/indictrans2-indic-en-dist-200M",
    "en_indic": "ai4bharat/indictrans2-en-indic-dist-200M",
}

print("Loading IndicTrans2 models...")

tokenizers = {}
translation_models = {}

for name, model_id in MODEL_IDS.items():
    print(f"Loading {model_id}")
    tokenizers[name] = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )
    translation_models[name] = AutoModelForSeq2SeqLM.from_pretrained(
        model_id,
        trust_remote_code=True
    ).to(device)

INDIC_LANGS = {
    "as","bn","brx","doi","gu","hi","kn","ks","kok",
    "mai","ml","mni","mr","ne","or","pa","sa",
    "sat","sd","ta","te","ur"
}

TAG_MAP = {
    "as":"asm_Beng",
    "bn":"ben_Beng",
    "brx":"brx_Deva",
    "doi":"doi_Deva",
    "en":"eng_Latn",
    "gu":"guj_Gujr",
    "hi":"hin_Deva",
    "kn":"kan_Knda",
    "ks":"kas_Arab",
    "kok":"gom_Deva",
    "mai":"mai_Deva",
    "ml":"mal_Mlym",
    "mni":"mni_Beng",
    "mr":"mar_Deva",
    "ne":"npi_Deva",
    "or":"ory_Orya",
    "pa":"pan_Guru",
    "sa":"san_Deva",
    "sat":"sat_Olck",
    "sd":"snd_Arab",
    "ta":"tam_Taml",
    "te":"tel_Telu",
    "ur":"urd_Arab"
}

def translate(text, source_lang, target_lang):
    if source_lang == target_lang:
        return text

    if source_lang == "en" and target_lang in INDIC_LANGS:
        model_key = "en_indic"
    elif source_lang in INDIC_LANGS and target_lang == "en":
        model_key = "indic_en"
    elif source_lang in INDIC_LANGS and target_lang in INDIC_LANGS:
        model_key = "indic_indic"
    else:
        raise ValueError(f"Unsupported translation: {source_lang} -> {target_lang}")

    tokenizer = tokenizers[model_key]
    model = translation_models[model_key]

    src_tag = TAG_MAP[source_lang]
    tgt_tag = TAG_MAP[target_lang]

    inputs = tokenizer(
        text,
        src_lang=src_tag,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            tgt_lang=tgt_tag,
            max_new_tokens=512
        )

    return tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )[0]

print(f"Detected language: {language}")
print(result)

target_language=input("Enter the language to which i need to get translated")

translated_text = translate(result, language, target_language)

print("\n==============================")
print(f"Translation ({language} -> {target_language})")
print("==============================")
print(translated_text)